# MetaseedClient Demo

This notebook demonstrates the programmatic API for working with MIAPPE-compliant metadata using `MetaseedClient`.

**When to use MetaseedClient vs ProfileFacade:**
- **MetaseedClient**: Scripts, web apps, CLIs - clean programmatic access
- **ProfileFacade**: Jupyter notebooks - interactive exploration with tab completion

## 1. Creating a Client

Always specify profile and version explicitly.

In [1]:
from metaseed import MetaseedClient

# Create client with explicit profile and version
client = MetaseedClient(profile="miappe", version="1.2")

print(client)
print(f"Profile: {client.profile}")
print(f"Version: {client.version}")

<MetaseedClient: miappe v1.2>
Profile: miappe
Version: 1.2


## 2. Schema Introspection

Explore available entity types and their fields.

In [2]:
# List all entity types
entity_types = client.list_entity_types()
print(f"MIAPPE 1.2 has {len(entity_types)} entities:")
for et in entity_types:
    print(f"  - {et}")

MIAPPE 1.2 has 14 entities:
  - Investigation
  - Study
  - Person
  - BiologicalMaterial
  - ObservationUnit
  - ObservedVariable
  - Factor
  - FactorValue
  - Event
  - Environment
  - Sample
  - DataFile
  - Location
  - MaterialSource


In [3]:
# Get fields for an entity type
fields = client.get_entity_fields("Investigation")
print("Investigation fields:")
for f in fields:
    req = "(required)" if f.required else ""
    print(f"  {f.name}: {f.type} {req}")

Investigation fields:
  unique_id: string (required)
  title: string (required)
  description: string 
  submission_date: date 
  public_release_date: date 
  license: uri 
  miappe_version: string 
  associated_publications: list 
  contacts: list 
  studies: list 


In [4]:
# Get complete schema info
schema = client.get_entity_schema("Study")
print(f"Entity: {schema.name}")
print(f"Description: {schema.description[:80]}...")
print(f"Ontology: {schema.ontology_term}")
print(f"Required fields: {schema.required_fields}")

Entity: Study
Description: A study represents a single experiment within an investigation.
...
Ontology: PPEO:study
Required fields: ('unique_id', 'investigation_id', 'title')


## 3. Creating Entities

Create entities with `create_entity()`. In MIAPPE 1.2, entities use flat references (`investigation_id`, `study_id`) to link to parents.

In [5]:
# Create an Investigation
inv = client.create_entity(
    "Investigation",
    {
        "unique_id": "INV-WHEAT-2024",
        "title": "Wheat Drought Tolerance Study",
        "description": "Multi-environment drought stress assessment",
        "miappe_version": "1.2",
    },
)

print(f"Created: {inv.entity_type}")
print(f"  ID: {inv.id}")
print(f"  unique_id: {inv.data['unique_id']}")
print(f"  title: {inv.data['title']}")

Created: Investigation
  ID: 3acae317
  unique_id: INV-WHEAT-2024
  title: Wheat Drought Tolerance Study


In [6]:
# Create a Study with reference to Investigation
study = client.create_entity(
    "Study",
    {
        "unique_id": "STU-FIELD-2024",
        "investigation_id": "INV-WHEAT-2024",  # Reference to parent
        "title": "Field Trial Germany 2024",
        "description": "Randomized block design with 3 replicates",
        "latitude": 52.52,
        "longitude": 13.405,
        "growth_facility_type": "field",
    },
    parent_id=inv.id,  # Links in entity tree
)

print(f"Created: {study.entity_type}")
print(f"  ID: {study.id}")
print(f"  investigation_id: {study.data['investigation_id']}")
print(f"  parent_id: {study.parent_id}")

Created: Study
  ID: 1281324f
  investigation_id: INV-WHEAT-2024
  parent_id: 3acae317


In [7]:
# Create BiologicalMaterials with study_id reference
materials = []
for i, cultivar in enumerate(["Apache", "Soissons", "Bologna"], start=1):
    mat = client.create_entity(
        "BiologicalMaterial",
        {
            "unique_id": f"BM-{i:03d}",
            "study_id": "STU-FIELD-2024",  # Reference to parent Study
            "organism": "Triticum aestivum",
            "genus": "Triticum",
            "species": "aestivum",
            "infraspecific_name": f"cv. {cultivar}",
        },
        parent_id=study.id,
    )
    materials.append(mat)
    print(f"Created: {mat.data['unique_id']} - {mat.data['infraspecific_name']}")

Created: BM-001 - cv. Apache
Created: BM-002 - cv. Soissons
Created: BM-003 - cv. Bologna


## 4. Entity Tree Navigation

Navigate the entity hierarchy.

In [8]:
# Get entity tree
tree = client.get_tree()

print("Entity Tree:")
for root in tree:
    print(f"{root.entity_type}: {root.label}")
    for child in root.children:
        print(f"  {child.entity_type}: {child.label}")
        for grandchild in child.children:
            print(f"    {grandchild.entity_type}: {grandchild.label}")

Entity Tree:
Investigation: INV-WHEAT-2024
  Study: STU-FIELD-2024
    BiologicalMaterial: BM-001
    BiologicalMaterial: BM-002
    BiologicalMaterial: BM-003


In [9]:
# Get root entities
roots = client.get_roots()
print(f"Root entities: {len(roots)}")
for r in roots:
    print(f"  {r.entity_type}: {r.label}")

Root entities: 1
  Investigation: INV-WHEAT-2024


In [10]:
# Get children of a specific entity
children = client.get_children(study.id)
print(f"Children of {study.data['unique_id']}:")
for c in children:
    print(f"  {c.entity_type}: {c.label}")

Children of STU-FIELD-2024:
  BiologicalMaterial: BM-001
  BiologicalMaterial: BM-002
  BiologicalMaterial: BM-003


## 5. Updating and Deleting Entities

In [11]:
# Get entity by ID
entity = client.get_entity(inv.id)
print(f"Retrieved: {entity.data['title']}")

Retrieved: Wheat Drought Tolerance Study


In [12]:
# Update an entity
updated = client.update_entity(
    inv.id,
    {
        "unique_id": "INV-WHEAT-2024",
        "title": "Wheat Drought Tolerance Study - Updated",
        "description": "Updated description with more details",
        "miappe_version": "1.2",
    },
)
print(f"Updated title: {updated.data['title']}")

Updated title: Wheat Drought Tolerance Study - Updated


Delete an entity (and its children) with:

```python
client.delete_entity(materials[2].id)
```

## 6. Validation

Validate individual entities or the entire dataset.

In [ ]:
# Validate all entities
result = client.validate()

if result.valid:
    print(f"All {result.entity_count} entities are valid!")
else:
    print(f"Found {result.error_count} validation issues:")
    for issue in result.issues:
        # ValidationIssue has: field, message, rule
        print(f"  {issue.field}: {issue.message}")

In [15]:
# Validate a specific entity
entity_result = client.validate_entity(study.id)
print(f"Study valid: {entity_result.valid}")

Study valid: True


## 7. Serialization

Export entities to different formats.

In [ ]:
# Serialize to flat format
flat_data = client.serialize(format="flat")

print(f"Profile: {flat_data['profile']}")
print(f"Version: {flat_data['version']}")
print(f"Entities: {len(flat_data['entities'])}")
print("\nEntity IDs:")
for e in flat_data["entities"]:
    # Flat format uses _type, and data fields are at top level
    print(f"  {e['_type']}: {e.get('unique_id', 'N/A')}")

In [17]:
# Serialize to tree format
tree_data = client.serialize(format="tree")

print(f"Root entities: {len(tree_data['tree'])}")
print("\nTree structure:")
for root in tree_data["tree"]:
    print(f"{root['entity_type']}: {root['data'].get('unique_id')}")
    for child in root.get("children", []):
        print(f"  {child['entity_type']}: {child['data'].get('unique_id')}")

Root entities: 1

Tree structure:
Investigation: INV-WHEAT-2024
  Study: STU-FIELD-2024


In [ ]:
# Save to YAML file
from pathlib import Path
import tempfile
import yaml

temp_dir = Path(tempfile.mkdtemp())
output_path = temp_dir / "dataset.yaml"

# Serialize and write manually
data = client.serialize(format="flat")
with open(output_path, "w") as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(f"Saved to: {output_path}")
print("\n" + "=" * 60)
print(output_path.read_text()[:1000] + "...")

## 8. Loading from Files

In [ ]:
# Create a new client and load the saved data
new_client = MetaseedClient(profile="miappe", version="1.2")

# Load from YAML file
with open(output_path) as f:
    saved_data = yaml.safe_load(f)
new_client.load(saved_data)

# Verify loaded data
loaded_tree = new_client.get_tree()
print(f"Loaded {len(loaded_tree)} root entities")
for root in loaded_tree:
    print(f"  {root.entity_type}: {root.label}")

## 9. Working with Pydantic Models

Access the underlying Pydantic models for advanced use cases.

In [ ]:
# Get the Pydantic model class
InvestigationModel = client.get_model("Investigation")

print(f"Model: {InvestigationModel.__name__}")
print(f"Fields: {list(InvestigationModel.model_fields.keys())[:5]}...")

# Create instance directly
direct_inv = InvestigationModel(
    unique_id="INV-DIRECT",
    title="Created with Pydantic model directly",
)
print(f"\nDirect instance: {direct_inv.title}")

In [ ]:
# Export JSON schema
import json

schema = InvestigationModel.model_json_schema()
print("JSON Schema (excerpt):")
print(
    json.dumps(
        {k: v for k, v in schema.items() if k in ["title", "required", "properties"]},
        indent=2,
    )[:500]
)

## Summary

MetaseedClient provides:

1. **Schema introspection**: `list_entity_types()`, `get_entity_fields()`, `get_entity_schema()`
2. **CRUD operations**: `create_entity()`, `get_entity()`, `update_entity()`, `delete_entity()`
3. **Tree navigation**: `get_tree()`, `get_roots()`, `get_children()`
4. **Validation**: `validate()`, `validate_entity()`
5. **Serialization**: `serialize()`, `load()`, `load_yaml()`, `clear()`
6. **Model access**: `get_model()` for Pydantic models

**MIAPPE 1.2**: Uses flat references (`investigation_id`, `study_id`) rather than nesting.

For interactive exploration in Jupyter, use `ProfileFacade` instead (via `miappe()`, `isa()`, etc.).